<a href="https://colab.research.google.com/github/SANGHATI23/neurofhir-qc/blob/main/00_NeuroFHIR_QC_Project_Foundation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# NeuroFHIR-QC
# Notebook 00: Project Foundation
# Cell 1: Mount Google Drive and create the project structure
# ============================================================

from google.colab import drive
from pathlib import Path
from datetime import datetime, timezone
import json
import platform
import sys

# Mount Google Drive so the project survives Colab disconnections.
drive.mount("/content/drive", force_remount=False)

# Permanent project location.
PROJECT_ROOT = Path("/content/drive/MyDrive/neurofhir-qc")

# Competition-oriented repository structure.
PROJECT_DIRECTORIES = [
    "notebooks",
    "data/synthetic_fhir",
    "data/sample_biomarkers",
    "data/sample_images",
    "data/sample_masks",
    "backend/app/api",
    "backend/app/services",
    "backend/app/fhir_builders",
    "evaluation/results",
    "docs",
    "scripts",
    "tests/unit",
    "tests/integration",
    "submission/fhir_resources",
    "submission/screenshots",
]

for directory in PROJECT_DIRECTORIES:
    (PROJECT_ROOT / directory).mkdir(parents=True, exist_ok=True)

# Lock the scope now so the project does not expand uncontrollably.
project_config = {
    "project_name": "NeuroFHIR-QC",
    "version": "0.1.0",
    "competition": "AMIA/HL7 FHIR App Competition",
    "submission_positioning": "Academic research implementation",
    "clinical_scope": (
        "Longitudinal brain-tumor or lesion volume tracking "
        "from baseline and follow-up MRI"
    ),
    "data_environment": (
        "Public de-identified brain MRI data and synthetic FHIR R4 records only"
    ),
    "core_fhir_resources": [
        "Patient",
        "Condition",
        "ImagingStudy",
        "Observation",
        "DiagnosticReport",
        "Device",
        "Provenance",
        "Task",
        "Bundle",
    ],
    "required_demo_cases": [
        "stable",
        "progression",
        "low-confidence",
    ],
    "safety_rule": (
        "Every AI-generated Observation begins as preliminary and cannot "
        "become final without an explicit human-review decision."
    ),
    "competition_moment": (
        "Detect an unstable segmentation, keep the result preliminary, "
        "route it for human review, and preserve the FHIR provenance."
    ),
    "created_utc": datetime.now(timezone.utc).isoformat(),
}

config_path = PROJECT_ROOT / "project_config.json"

with config_path.open("w", encoding="utf-8") as file:
    json.dump(project_config, file, indent=2)

# Verify that the foundation was created successfully.
assert PROJECT_ROOT.exists(), "Project root was not created."
assert config_path.exists(), "Project configuration was not created."

print("=" * 70)
print("✅ NeuroFHIR-QC project foundation created successfully")
print(f"📁 Project root: {PROJECT_ROOT}")
print(f"⚙️ Configuration: {config_path}")
print(f"🐍 Python version: {sys.version.split()[0]}")
print(f"💻 Runtime platform: {platform.platform()}")
print(f"📂 Directories created: {len(PROJECT_DIRECTORIES)}")
print("🎯 Locked scope: longitudinal brain-tumor/lesion volume + QC + FHIR review")
print("=" * 70)

Mounted at /content/drive
✅ NeuroFHIR-QC project foundation created successfully
📁 Project root: /content/drive/MyDrive/neurofhir-qc
⚙️ Configuration: /content/drive/MyDrive/neurofhir-qc/project_config.json
🐍 Python version: 3.12.13
💻 Runtime platform: Linux-6.6.122+-x86_64-with-glibc2.35
📂 Directories created: 15
🎯 Locked scope: longitudinal brain-tumor/lesion volume + QC + FHIR review


In [2]:
# ============================================================
# NeuroFHIR-QC
# Notebook 00: Project Foundation
# Cell 2: Create repository metadata and validate locked scope
# ============================================================

from pathlib import Path
import json
import textwrap

PROJECT_ROOT = Path("/content/drive/MyDrive/neurofhir-qc")
CONFIG_PATH = PROJECT_ROOT / "project_config.json"

# ------------------------------------------------------------
# 1. Load and validate the project configuration
# ------------------------------------------------------------

if not CONFIG_PATH.exists():
    raise FileNotFoundError(
        f"Project configuration not found: {CONFIG_PATH}. "
        "Run Cell 1 before running this cell."
    )

with CONFIG_PATH.open("r", encoding="utf-8") as file:
    config = json.load(file)

required_config_fields = {
    "project_name",
    "version",
    "competition",
    "submission_positioning",
    "clinical_scope",
    "data_environment",
    "core_fhir_resources",
    "required_demo_cases",
    "safety_rule",
    "competition_moment",
}

missing_fields = required_config_fields.difference(config.keys())

assert not missing_fields, (
    "The project configuration is missing required fields: "
    f"{sorted(missing_fields)}"
)

required_fhir_resources = {
    "Patient",
    "Condition",
    "ImagingStudy",
    "Observation",
    "DiagnosticReport",
    "Device",
    "Provenance",
    "Task",
    "Bundle",
}

configured_resources = set(config["core_fhir_resources"])

missing_resources = required_fhir_resources.difference(configured_resources)

assert not missing_resources, (
    "Required FHIR resources are missing from the configuration: "
    f"{sorted(missing_resources)}"
)

required_demo_cases = {
    "stable",
    "progression",
    "low-confidence",
}

configured_demo_cases = set(config["required_demo_cases"])

assert configured_demo_cases == required_demo_cases, (
    "The competition demo must remain limited to exactly these cases: "
    f"{sorted(required_demo_cases)}"
)

assert config["project_name"] == "NeuroFHIR-QC"
assert "longitudinal" in config["clinical_scope"].lower()
assert "tumor" in config["clinical_scope"].lower() or "lesion" in config["clinical_scope"].lower()
assert "synthetic fhir" in config["data_environment"].lower()
assert "preliminary" in config["safety_rule"].lower()
assert "human-review" in config["competition_moment"].lower()

# ------------------------------------------------------------
# 2. Create README.md
# ------------------------------------------------------------

readme_content = textwrap.dedent(
    f"""
    # {config["project_name"]}

    **{config["competition"]} submission project**

    NeuroFHIR-QC is a human-in-the-loop FHIR R4 application for converting
    longitudinal brain-tumor or lesion measurements into quality-scored,
    provenance-aware and reviewable FHIR evidence.

    ## Submission Positioning

    {config["submission_positioning"]}

    The application is evaluated using public de-identified brain MRI data and
    synthetic FHIR R4 records. It is not presented as a clinically deployed,
    diagnostically validated or hospital-integrated system.

    ## Locked MVP Scope

    {config["clinical_scope"]}

    The competition demonstration contains three cases:

    1. Stable longitudinal volume
    2. Progression with a reliable model output
    3. Low-confidence or unstable output requiring human review

    ## Core Competition Workflow

    1. Load synthetic FHIR patient and imaging context.
    2. Retrieve baseline and follow-up neuroimaging information.
    3. calculate or import lesion/tumor volume.
    4. calculate longitudinal volume change.
    5. calculate quality-control and provenance indicators.
    6. keep the AI result preliminary until human review.
    7. allow accept, reject or correction-required decisions.
    8. generate linked FHIR R4 resources.
    9. write a transaction Bundle to a FHIR server.
    10. display FHIR JSON, references, provenance and validation results.

    ## Core FHIR Resources

    {", ".join(config["core_fhir_resources"])}

    ## Safety Rule

    {config["safety_rule"]}

    ## Competition Demonstration Moment

    {config["competition_moment"]}

    ## Explicitly Outside Version 1

    - Autonomous diagnosis
    - Treatment recommendations
    - Alzheimer disease workflows
    - Parkinson disease workflows
    - Stroke workflows
    - Genomics
    - OMOP conversion
    - Hospital production deployment
    - Epic, Cerner or PACS production integration
    - Clinical safety or clinical validation claims

    ## Repository Status

    Current phase: **Phase 1 — FHIR foundation**

    Version: `{config["version"]}`
    """
).strip() + "\n"

README_PATH = PROJECT_ROOT / "README.md"
README_PATH.write_text(readme_content, encoding="utf-8")

# ------------------------------------------------------------
# 3. Create the formal scope-lock document
# ------------------------------------------------------------

scope_lock_content = textwrap.dedent(
    """
    # NeuroFHIR-QC Scope Lock

    ## Competition MVP

    NeuroFHIR-QC will demonstrate one complete longitudinal brain MRI workflow:

    - synthetic patient context;
    - baseline and follow-up ImagingStudy records;
    - lesion or tumor volume;
    - longitudinal percentage change;
    - quality-control indicators;
    - provenance completeness;
    - human accept, reject or correction-required workflow;
    - validated FHIR transaction Bundle;
    - visible FHIR audit trail.

    ## Required Demonstration Cases

    ### Case 1: Stable

    The baseline and follow-up measurements remain similar. Quality indicators
    are acceptable, the reviewer accepts the result and the Observation can
    become final.

    ### Case 2: Progression

    Follow-up volume is meaningfully larger than baseline. Quality indicators
    remain acceptable, longitudinal change is displayed and the reviewer
    accepts the result.

    ### Case 3: Low Confidence

    The output is unstable, incomplete or implausible. The application displays
    a warning, keeps the Observation preliminary and routes the result for
    rejection or correction.

    ## Non-Negotiable Safety Behavior

    No AI-generated Observation may become final without an explicit human
    review action.

    ## Version 1 Exclusions

    Version 1 will not include:

    - autonomous clinical diagnosis;
    - treatment recommendations;
    - multiple neurological disease modules;
    - genomics;
    - OMOP;
    - Bulk FHIR;
    - CDS Hooks;
    - CQL;
    - real patient data;
    - hospital deployment claims.

    These exclusions protect the competition build from uncontrolled scope
    expansion.
    """
).strip() + "\n"

SCOPE_PATH = PROJECT_ROOT / "docs" / "SCOPE_LOCK.md"
SCOPE_PATH.write_text(scope_lock_content, encoding="utf-8")

# ------------------------------------------------------------
# 4. Create a safe initial .gitignore
# ------------------------------------------------------------

gitignore_content = textwrap.dedent(
    """
    # Python
    __pycache__/
    *.py[cod]
    *.pyo
    .pytest_cache/
    .mypy_cache/

    # Jupyter / Colab
    .ipynb_checkpoints/

    # Virtual environments
    .venv/
    venv/
    env/

    # Environment variables and secrets
    .env
    .env.*
    !.env.example
    *.pem
    *.key

    # Operating system files
    .DS_Store
    Thumbs.db

    # Raw or private data
    data/raw/
    data/private/
    data/downloads/

    # Model checkpoints
    model/checkpoints/
    models/checkpoints/
    *.pt
    *.pth
    *.ckpt

    # Runtime outputs
    logs/
    tmp/
    temp/

    # Frontend dependencies and builds
    frontend/node_modules/
    frontend/dist/
    """
).strip() + "\n"

GITIGNORE_PATH = PROJECT_ROOT / ".gitignore"
GITIGNORE_PATH.write_text(gitignore_content, encoding="utf-8")

# ------------------------------------------------------------
# 5. Final verification
# ------------------------------------------------------------

created_files = [
    README_PATH,
    SCOPE_PATH,
    GITIGNORE_PATH,
]

for path in created_files:
    assert path.exists(), f"Expected file was not created: {path}"
    assert path.stat().st_size > 0, f"Created file is empty: {path}"

print("=" * 72)
print("✅ Repository metadata created successfully")
print(f"✅ README created: {README_PATH}")
print(f"✅ Scope lock created: {SCOPE_PATH}")
print(f"✅ Git ignore created: {GITIGNORE_PATH}")
print(f"✅ Required FHIR resources validated: {len(required_fhir_resources)}")
print(f"✅ Required demo cases validated: {len(required_demo_cases)}")
print("🔒 Scope validation passed")
print("🎯 NeuroFHIR-QC remains locked to the competition-ready MVP")
print("=" * 72)

AssertionError: 

In [3]:
# ============================================================
# NeuroFHIR-QC
# Notebook 00: Project Foundation
# Cell 2: Create repository metadata and validate locked scope
# ============================================================

from pathlib import Path
import json
import re
import textwrap


# ------------------------------------------------------------
# 1. Define project paths
# ------------------------------------------------------------

PROJECT_ROOT = Path("/content/drive/MyDrive/neurofhir-qc")
CONFIG_PATH = PROJECT_ROOT / "project_config.json"

README_PATH = PROJECT_ROOT / "README.md"
SCOPE_PATH = PROJECT_ROOT / "docs" / "SCOPE_LOCK.md"
GITIGNORE_PATH = PROJECT_ROOT / ".gitignore"


# ------------------------------------------------------------
# 2. Confirm that Cell 1 was completed
# ------------------------------------------------------------

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        f"Project root was not found: {PROJECT_ROOT}\n"
        "Run Notebook 00, Cell 1 before running this cell."
    )

if not CONFIG_PATH.exists():
    raise FileNotFoundError(
        f"Project configuration was not found: {CONFIG_PATH}\n"
        "Run Notebook 00, Cell 1 before running this cell."
    )


# ------------------------------------------------------------
# 3. Load project configuration
# ------------------------------------------------------------

with CONFIG_PATH.open("r", encoding="utf-8") as file:
    config = json.load(file)


# ------------------------------------------------------------
# 4. Validate required configuration fields
# ------------------------------------------------------------

required_config_fields = {
    "project_name",
    "version",
    "competition",
    "submission_positioning",
    "clinical_scope",
    "data_environment",
    "core_fhir_resources",
    "required_demo_cases",
    "safety_rule",
    "competition_moment",
    "created_utc",
}

missing_config_fields = required_config_fields.difference(config.keys())

assert not missing_config_fields, (
    "The project configuration is missing required fields: "
    f"{sorted(missing_config_fields)}"
)


# ------------------------------------------------------------
# 5. Validate the required FHIR resources
# ------------------------------------------------------------

required_fhir_resources = {
    "Patient",
    "Condition",
    "ImagingStudy",
    "Observation",
    "DiagnosticReport",
    "Device",
    "Provenance",
    "Task",
    "Bundle",
}

configured_fhir_resources = set(config["core_fhir_resources"])

missing_fhir_resources = required_fhir_resources.difference(
    configured_fhir_resources
)

assert not missing_fhir_resources, (
    "The following required FHIR resources are missing: "
    f"{sorted(missing_fhir_resources)}"
)


# ------------------------------------------------------------
# 6. Validate the three competition demonstration cases
# ------------------------------------------------------------

required_demo_cases = {
    "stable",
    "progression",
    "low-confidence",
}

configured_demo_cases = set(config["required_demo_cases"])

assert configured_demo_cases == required_demo_cases, (
    "The competition demonstration must contain exactly these cases: "
    f"{sorted(required_demo_cases)}"
)


# ------------------------------------------------------------
# 7. Normalize text before scope validation
# ------------------------------------------------------------

def normalize_text(value: str) -> str:
    """
    Convert capitalization, hyphens, underscores and repeated spaces
    into a consistent format for reliable validation.
    """
    normalized = value.lower()
    normalized = re.sub(r"[-_]+", " ", normalized)
    normalized = re.sub(r"\s+", " ", normalized)
    return normalized.strip()


normalized_scope = normalize_text(config["clinical_scope"])
normalized_environment = normalize_text(config["data_environment"])
normalized_safety_rule = normalize_text(config["safety_rule"])
normalized_competition_moment = normalize_text(
    config["competition_moment"]
)


# ------------------------------------------------------------
# 8. Validate the locked competition scope
# ------------------------------------------------------------

assert config["project_name"] == "NeuroFHIR-QC", (
    "Unexpected project name."
)

assert "longitudinal" in normalized_scope, (
    "The clinical scope must remain longitudinal."
)

assert (
    "tumor" in normalized_scope
    or "lesion" in normalized_scope
), (
    "The clinical scope must include tumor or lesion volume."
)

assert "synthetic fhir" in normalized_environment, (
    "The data environment must specify synthetic FHIR records."
)

assert "public de identified" in normalized_environment, (
    "The data environment must specify public de-identified imaging data."
)

assert "preliminary" in normalized_safety_rule, (
    "The safety rule must keep AI-generated results preliminary."
)

assert "human review" in normalized_safety_rule, (
    "The safety rule must require explicit human review."
)

assert "unstable segmentation" in normalized_competition_moment, (
    "The competition moment must demonstrate unstable segmentation detection."
)

assert "human review" in normalized_competition_moment, (
    "The competition moment must include human review."
)

assert "fhir provenance" in normalized_competition_moment, (
    "The competition moment must preserve FHIR provenance."
)


# ------------------------------------------------------------
# 9. Create README.md
# ------------------------------------------------------------

readme_content = textwrap.dedent(
    f"""
    # {config["project_name"]}

    **Competition:** {config["competition"]}

    **Version:** {config["version"]}

    **Submission positioning:** {config["submission_positioning"]}

    ## Project Summary

    NeuroFHIR-QC is a human-in-the-loop FHIR R4 application for
    transforming longitudinal brain-tumor or lesion measurements into
    quality-scored, provenance-aware, reviewable and reusable FHIR
    evidence.

    The project is evaluated using public de-identified brain MRI data
    and synthetic FHIR R4 records.

    It is not presented as:

    - a clinically deployed application;
    - an autonomous diagnostic system;
    - a treatment-recommendation system;
    - a hospital production implementation;
    - a clinically validated medical device.

    ## Locked MVP Scope

    {config["clinical_scope"]}

    ## Required Demonstration Cases

    1. Stable longitudinal tumor or lesion volume
    2. Progression with a reliable model output
    3. Low-confidence or unstable output requiring human review

    ## Core Competition Workflow

    1. Load a synthetic FHIR patient.
    2. Retrieve Patient, Condition and ImagingStudy context.
    3. Retrieve a previous tumor or lesion volume Observation.
    4. Load baseline and follow-up neuroimaging inputs.
    5. Calculate or import the follow-up volume.
    6. Calculate absolute and percentage longitudinal change.
    7. Calculate quality-control and provenance indicators.
    8. Keep the AI-generated Observation preliminary.
    9. Allow accept, reject or correction-required decisions.
    10. Generate linked FHIR R4 resources.
    11. Create a FHIR transaction Bundle.
    12. Write the Bundle to a FHIR server.
    13. Display FHIR JSON, resource references and validation results.

    ## Core FHIR Resources

    - Patient
    - Condition
    - ImagingStudy
    - Observation
    - DiagnosticReport
    - Device
    - Provenance
    - Task
    - Bundle

    ## Human-in-the-Loop Safety Rule

    {config["safety_rule"]}

    ## Competition Demonstration Moment

    {config["competition_moment"]}

    ## Explicitly Outside Version 1

    The following are excluded from the competition MVP:

    - autonomous diagnosis;
    - treatment recommendations;
    - Alzheimer disease workflows;
    - Parkinson disease workflows;
    - stroke workflows;
    - traumatic brain injury workflows;
    - genomics;
    - OMOP conversion;
    - Bulk FHIR;
    - CDS Hooks;
    - CQL;
    - real patient data;
    - Epic production integration;
    - Cerner production integration;
    - PACS production integration;
    - hospital deployment claims;
    - clinical safety claims;
    - clinical validation claims.

    ## Current Build Phase

    **Phase 1: FHIR foundation**

    The current objective is to create reproducible synthetic FHIR
    resources and verify reliable read operations before implementing
    imaging inference or frontend functionality.
    """
).strip() + "\n"

README_PATH.write_text(
    readme_content,
    encoding="utf-8",
)


# ------------------------------------------------------------
# 10. Create formal scope-lock documentation
# ------------------------------------------------------------

scope_lock_content = textwrap.dedent(
    """
    # NeuroFHIR-QC Scope Lock

    ## Purpose

    This document prevents uncontrolled expansion of the AMIA/HL7 FHIR
    App Competition build.

    NeuroFHIR-QC will implement one complete, reproducible and measurable
    longitudinal brain MRI workflow.

    ## Competition MVP

    The MVP will contain:

    - synthetic FHIR patient context;
    - brain-tumor or lesion Condition context;
    - baseline ImagingStudy;
    - follow-up ImagingStudy;
    - previous tumor or lesion volume Observation;
    - current AI-generated volume Observation;
    - longitudinal absolute volume change;
    - longitudinal percentage volume change;
    - quality-control indicators;
    - provenance-completeness indicators;
    - human accept, reject and correction-required decisions;
    - linked DiagnosticReport;
    - versioned Device resource;
    - algorithmic and human Provenance resources;
    - human-review Task;
    - validated FHIR transaction Bundle;
    - visible FHIR JSON and reference audit trail.

    ## Required Demonstration Cases

    ### Case 1: Stable

    Baseline and follow-up measurements remain similar.

    Expected behavior:

    - quality indicators are acceptable;
    - the result begins as preliminary;
    - the reviewer accepts the result;
    - the Observation becomes final;
    - the Task becomes completed;
    - Provenance records the review action.

    ### Case 2: Progression

    The follow-up tumor or lesion volume is meaningfully larger than the
    baseline volume.

    Expected behavior:

    - longitudinal change is displayed;
    - quality indicators remain acceptable;
    - the result begins as preliminary;
    - the reviewer accepts the result;
    - the Observation becomes final;
    - the review action is recorded.

    ### Case 3: Low Confidence

    The AI output is unstable, incomplete or implausible.

    Expected behavior:

    - a visible warning is displayed;
    - the Observation remains preliminary;
    - finalization is blocked;
    - a Task routes the result for review;
    - the reviewer rejects the result or requests correction;
    - the original AI output remains traceable;
    - the complete provenance history is preserved.

    ## Non-Negotiable Safety Behavior

    No AI-generated Observation may become final without an explicit
    human-review decision.

    ## Required Competition Evidence

    Before submission, the project must produce measured evidence for:

    - segmentation performance;
    - robustness behavior;
    - FHIR validation;
    - FHIR transaction success;
    - internal reference integrity;
    - workflow timing;
    - human-review state transitions;
    - low-confidence result detection;
    - usability testing.

    ## Version 1 Exclusions

    Version 1 will not include:

    - autonomous clinical diagnosis;
    - autonomous treatment recommendations;
    - multiple neurological disease modules;
    - Alzheimer disease workflows;
    - Parkinson disease workflows;
    - stroke workflows;
    - genomics;
    - OMOP;
    - Bulk FHIR;
    - CDS Hooks;
    - CQL;
    - real patient data;
    - hospital production deployment;
    - unsupported clinical impact claims.

    These exclusions protect the competition build from uncontrolled
    scope expansion.
    """
).strip() + "\n"

SCOPE_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)

SCOPE_PATH.write_text(
    scope_lock_content,
    encoding="utf-8",
)


# ------------------------------------------------------------
# 11. Create a safe .gitignore file
# ------------------------------------------------------------

gitignore_content = textwrap.dedent(
    """
    # --------------------------------------------------------
    # Python
    # --------------------------------------------------------
    __pycache__/
    *.py[cod]
    *.pyo
    *.pyd
    .pytest_cache/
    .mypy_cache/
    .ruff_cache/

    # --------------------------------------------------------
    # Jupyter and Colab
    # --------------------------------------------------------
    .ipynb_checkpoints/

    # --------------------------------------------------------
    # Virtual environments
    # --------------------------------------------------------
    .venv/
    venv/
    env/

    # --------------------------------------------------------
    # Environment variables and secrets
    # --------------------------------------------------------
    .env
    .env.*
    !.env.example
    *.pem
    *.key
    *.p12
    secrets.json

    # --------------------------------------------------------
    # Operating system files
    # --------------------------------------------------------
    .DS_Store
    Thumbs.db

    # --------------------------------------------------------
    # Private, raw or downloaded data
    # --------------------------------------------------------
    data/raw/
    data/private/
    data/downloads/

    # --------------------------------------------------------
    # Large model checkpoints
    # --------------------------------------------------------
    model/checkpoints/
    models/checkpoints/
    *.pt
    *.pth
    *.ckpt

    # --------------------------------------------------------
    # Runtime outputs
    # --------------------------------------------------------
    logs/
    tmp/
    temp/

    # --------------------------------------------------------
    # Frontend dependencies and builds
    # --------------------------------------------------------
    frontend/node_modules/
    frontend/dist/

    # --------------------------------------------------------
    # IDE configuration
    # --------------------------------------------------------
    .vscode/
    .idea/
    """
).strip() + "\n"

GITIGNORE_PATH.write_text(
    gitignore_content,
    encoding="utf-8",
)


# ------------------------------------------------------------
# 12. Verify all created files
# ------------------------------------------------------------

created_files = [
    README_PATH,
    SCOPE_PATH,
    GITIGNORE_PATH,
]

for created_file in created_files:
    assert created_file.exists(), (
        f"Expected file was not created: {created_file}"
    )

    assert created_file.is_file(), (
        f"Expected path is not a file: {created_file}"
    )

    assert created_file.stat().st_size > 0, (
        f"Created file is empty: {created_file}"
    )


# ------------------------------------------------------------
# 13. Display validation summary
# ------------------------------------------------------------

print("=" * 74)
print("✅ Repository metadata created successfully")
print(f"✅ README created: {README_PATH}")
print(f"✅ Scope lock created: {SCOPE_PATH}")
print(f"✅ Git ignore created: {GITIGNORE_PATH}")
print(
    "✅ Required FHIR resources validated: "
    f"{len(required_fhir_resources)}"
)
print(
    "✅ Required demonstration cases validated: "
    f"{len(required_demo_cases)}"
)
print("✅ Human-review safety rule validated")
print("✅ Low-confidence competition moment validated")
print("🔒 Scope validation passed")
print("🎯 NeuroFHIR-QC remains locked to the competition-ready MVP")
print("=" * 74)

✅ Repository metadata created successfully
✅ README created: /content/drive/MyDrive/neurofhir-qc/README.md
✅ Scope lock created: /content/drive/MyDrive/neurofhir-qc/docs/SCOPE_LOCK.md
✅ Git ignore created: /content/drive/MyDrive/neurofhir-qc/.gitignore
✅ Required FHIR resources validated: 9
✅ Required demonstration cases validated: 3
✅ Human-review safety rule validated
✅ Low-confidence competition moment validated
🔒 Scope validation passed
🎯 NeuroFHIR-QC remains locked to the competition-ready MVP


In [4]:
# ============================================================
# NeuroFHIR-QC
# Notebook 00: Project Foundation
# Cell 3: Create reproducibility and development metadata
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import json
import platform
import sys
import textwrap


# ------------------------------------------------------------
# 1. Define project paths
# ------------------------------------------------------------

PROJECT_ROOT = Path("/content/drive/MyDrive/neurofhir-qc")

REQUIREMENTS_DIR = PROJECT_ROOT / "requirements"
DOCS_DIR = PROJECT_ROOT / "docs"

ENV_EXAMPLE_PATH = PROJECT_ROOT / ".env.example"
LICENSE_PATH = PROJECT_ROOT / "LICENSE"
RUNTIME_MANIFEST_PATH = PROJECT_ROOT / "runtime_manifest.json"
NOTEBOOK_MANIFEST_PATH = PROJECT_ROOT / "notebook_manifest.json"
BUILD_PLAN_PATH = DOCS_DIR / "BUILD_PLAN.md"
REQUIREMENTS_README_PATH = REQUIREMENTS_DIR / "README.md"


# ------------------------------------------------------------
# 2. Confirm the project foundation exists
# ------------------------------------------------------------

required_existing_paths = [
    PROJECT_ROOT,
    PROJECT_ROOT / "project_config.json",
    PROJECT_ROOT / "README.md",
    PROJECT_ROOT / ".gitignore",
    PROJECT_ROOT / "docs" / "SCOPE_LOCK.md",
]

missing_existing_paths = [
    str(path)
    for path in required_existing_paths
    if not path.exists()
]

if missing_existing_paths:
    raise FileNotFoundError(
        "Notebook 00 foundation is incomplete. Missing paths:\n"
        + "\n".join(missing_existing_paths)
    )


# ------------------------------------------------------------
# 3. Create required directories
# ------------------------------------------------------------

REQUIREMENTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

DOCS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ------------------------------------------------------------
# 4. Create .env.example
# ------------------------------------------------------------

env_example_content = textwrap.dedent(
    """
    # ========================================================
    # NeuroFHIR-QC example environment configuration
    # ========================================================
    #
    # Copy this file to .env for local development.
    # Never commit real secrets, credentials or access tokens.
    #

    # Application environment
    APP_ENV=development
    APP_NAME=NeuroFHIR-QC
    APP_VERSION=0.1.0

    # Demo configuration
    DEMO_MODE=true
    SYNTHETIC_DATA_ONLY=true

    # FHIR server configuration
    HAPI_FHIR_BASE_URL=http://localhost:8080/fhir
    FHIR_VERSION=R4
    FHIR_REQUEST_TIMEOUT_SECONDS=30

    # Write-back safety control
    ALLOW_FHIR_WRITEBACK=false

    # Backend service
    BACKEND_HOST=0.0.0.0
    BACKEND_PORT=8000

    # Frontend service
    FRONTEND_PORT=5173

    # Logging
    LOG_LEVEL=INFO

    # SMART App Launch values will be configured later.
    SMART_CLIENT_ID=
    SMART_REDIRECT_URI=
    SMART_SCOPE=launch/patient patient/*.read patient/Observation.write patient/DiagnosticReport.write patient/Task.write
    """
).strip() + "\n"

ENV_EXAMPLE_PATH.write_text(
    env_example_content,
    encoding="utf-8",
)


# ------------------------------------------------------------
# 5. Create an open-source LICENSE
# ------------------------------------------------------------

license_content = textwrap.dedent(
    """
    MIT License

    Copyright (c) 2026 NeuroFHIR-QC contributors

    Permission is hereby granted, free of charge, to any person obtaining
    a copy of this software and associated documentation files
    (the "Software"), to deal in the Software without restriction,
    including without limitation the rights to use, copy, modify, merge,
    publish, distribute, sublicense, and/or sell copies of the Software,
    and to permit persons to whom the Software is furnished to do so,
    subject to the following conditions:

    The above copyright notice and this permission notice shall be
    included in all copies or substantial portions of the Software.

    THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND,
    EXPRESS OR IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF
    MERCHANTABILITY, FITNESS FOR A PARTICULAR PURPOSE AND
    NONINFRINGEMENT. IN NO EVENT SHALL THE AUTHORS OR COPYRIGHT HOLDERS
    BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER LIABILITY, WHETHER IN AN
    ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM, OUT OF OR IN
    CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
    SOFTWARE.
    """
).strip() + "\n"

LICENSE_PATH.write_text(
    license_content,
    encoding="utf-8",
)


# ------------------------------------------------------------
# 6. Record the current Colab runtime
# ------------------------------------------------------------

runtime_manifest = {
    "project_name": "NeuroFHIR-QC",
    "notebook": "00_NeuroFHIR_QC_Project_Foundation.ipynb",
    "runtime_type": "Google Colab",
    "python_version": platform.python_version(),
    "python_implementation": platform.python_implementation(),
    "platform": platform.platform(),
    "system": platform.system(),
    "machine": platform.machine(),
    "generated_utc": datetime.now(timezone.utc).isoformat(),
    "production_backend_target": {
        "python_version": "3.11",
        "reason": (
            "The competition backend and Docker environment will target "
            "Python 3.11 even when development notebooks run on the "
            "current Google Colab Python runtime."
        ),
    },
    "data_policy": {
        "public_deidentified_imaging_only": True,
        "synthetic_fhir_only": True,
        "real_patient_data_allowed": False,
    },
}

with RUNTIME_MANIFEST_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        runtime_manifest,
        file,
        indent=2,
    )


# ------------------------------------------------------------
# 7. Create the complete notebook build manifest
# ------------------------------------------------------------

notebook_manifest = {
    "project_name": "NeuroFHIR-QC",
    "version": "0.1.0",
    "generated_utc": datetime.now(timezone.utc).isoformat(),
    "execution_rule": (
        "Complete and validate each notebook before starting the next."
    ),
    "notebooks": [
        {
            "number": "00",
            "filename": (
                "00_NeuroFHIR_QC_Project_Foundation.ipynb"
            ),
            "purpose": (
                "Create repository structure, scope lock, environment "
                "metadata and foundation audit."
            ),
            "status": "in_progress",
        },
        {
            "number": "01",
            "filename": (
                "01_NeuroFHIR_QC_Synthetic_FHIR_Foundation.ipynb"
            ),
            "purpose": (
                "Create and validate synthetic Patient, Condition, "
                "ImagingStudy and prior Observation resources."
            ),
            "status": "not_started",
        },
        {
            "number": "02",
            "filename": (
                "02_NeuroFHIR_QC_HAPI_Server_and_Read_Operations.ipynb"
            ),
            "purpose": (
                "Seed a FHIR R4 server and verify patient-context "
                "read operations."
            ),
            "status": "not_started",
        },
        {
            "number": "03",
            "filename": (
                "03_NeuroFHIR_QC_Imaging_Data_Preparation.ipynb"
            ),
            "purpose": (
                "Prepare public de-identified MRI data, metadata, "
                "masks and reproducible preprocessing."
            ),
            "status": "not_started",
        },
        {
            "number": "04",
            "filename": (
                "04_NeuroFHIR_QC_Segmentation_and_Volumetry.ipynb"
            ),
            "purpose": (
                "Run segmentation, calculate lesion or tumor volume "
                "and generate overlay artifacts."
            ),
            "status": "not_started",
        },
        {
            "number": "05",
            "filename": (
                "05_NeuroFHIR_QC_Trust_and_Robustness_Engine.ipynb"
            ),
            "purpose": (
                "Calculate QC indicators, perturbation stability, "
                "plausibility and low-confidence triage."
            ),
            "status": "not_started",
        },
        {
            "number": "06",
            "filename": (
                "06_NeuroFHIR_QC_Longitudinal_Analysis.ipynb"
            ),
            "purpose": (
                "Calculate baseline-to-follow-up absolute and "
                "percentage volume changes."
            ),
            "status": "not_started",
        },
        {
            "number": "07",
            "filename": (
                "07_NeuroFHIR_QC_FHIR_Evidence_and_Writeback.ipynb"
            ),
            "purpose": (
                "Generate Observation, DiagnosticReport, Device, "
                "Provenance, Task and transaction Bundle resources."
            ),
            "status": "not_started",
        },
        {
            "number": "08",
            "filename": (
                "08_NeuroFHIR_QC_Human_Review_Workflow.ipynb"
            ),
            "purpose": (
                "Implement accept, reject and correction-required "
                "state transitions with review provenance."
            ),
            "status": "not_started",
        },
        {
            "number": "09",
            "filename": (
                "09_NeuroFHIR_QC_Evaluation.ipynb"
            ),
            "purpose": (
                "Measure segmentation, robustness, FHIR conformance, "
                "workflow timing and safety behavior."
            ),
            "status": "not_started",
        },
        {
            "number": "10",
            "filename": (
                "10_NeuroFHIR_QC_Competition_Artifacts.ipynb"
            ),
            "purpose": (
                "Export validation results, example Bundles, figures, "
                "tables and submission-ready evidence."
            ),
            "status": "not_started",
        },
    ],
}

with NOTEBOOK_MANIFEST_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        notebook_manifest,
        file,
        indent=2,
    )


# ------------------------------------------------------------
# 8. Create requirements documentation
# ------------------------------------------------------------

requirements_readme_content = textwrap.dedent(
    """
    # NeuroFHIR-QC Dependency Strategy

    Dependency files will be created and pinned incrementally as each
    validated project module is implemented.

    This avoids installing unnecessary libraries before they are needed
    and ensures that the final requirements reflect code that was
    actually executed.

    ## Planned Dependency Groups

    - `requirements-fhir.txt`
      - FHIR resource construction
      - HTTP communication
      - FHIR validation support

    - `requirements-imaging.txt`
      - NIfTI and DICOM handling
      - image preprocessing
      - segmentation and volumetry

    - `requirements-backend.txt`
      - FastAPI backend
      - configuration
      - testing

    - `requirements-evaluation.txt`
      - performance metrics
      - statistical evaluation
      - workflow timing

    - `requirements-dev.txt`
      - testing
      - formatting
      - static analysis

    ## Python Versions

    Google Colab notebooks may use the active Colab Python version.

    The production backend and Docker image will target Python 3.11,
    consistent with the locked implementation blueprint.

    ## Dependency Rule

    A package will not be added to a final pinned requirements file
    until:

    1. it is required by working code;
    2. the relevant notebook executes successfully;
    3. its installed version is recorded;
    4. compatibility is verified.
    """
).strip() + "\n"

REQUIREMENTS_README_PATH.write_text(
    requirements_readme_content,
    encoding="utf-8",
)


# ------------------------------------------------------------
# 9. Create the repository build plan
# ------------------------------------------------------------

build_plan_content = textwrap.dedent(
    """
    # NeuroFHIR-QC Build Plan

    ## Governing Principle

    Build one closed, reproducible and measurable longitudinal
    brain-tumor or lesion workflow completely.

    Do not broaden the disease scope until the competition MVP works
    end to end.

    ## Phase 1: FHIR Foundation

    Notebooks:

    - Notebook 00: Project foundation
    - Notebook 01: Synthetic FHIR resources
    - Notebook 02: FHIR server and read operations

    Exit criterion:

    A selected synthetic patient and all required imaging-context
    resources load reliably from a FHIR R4 server.

    ## Phase 2: Imaging Pipeline

    Notebooks:

    - Notebook 03: Imaging data preparation
    - Notebook 04: Segmentation and volumetry

    Exit criterion:

    One public de-identified MRI case runs reproducibly from source
    image to segmentation mask, volume measurement and overlay.

    ## Phase 3: Trust and Longitudinal Analysis

    Notebooks:

    - Notebook 05: Trust and robustness engine
    - Notebook 06: Longitudinal analysis

    Exit criterion:

    Stable, progression and low-confidence cases are differentiated
    using visible and reproducible metrics.

    ## Phase 4: FHIR Evidence and Human Review

    Notebooks:

    - Notebook 07: FHIR evidence and transaction write-back
    - Notebook 08: Human-review workflow

    Exit criterion:

    Linked resources validate, transaction write-back succeeds and
    accepted, rejected and correction-required states preserve their
    provenance.

    ## Phase 5: Evaluation and Competition Package

    Notebooks:

    - Notebook 09: Quantitative evaluation
    - Notebook 10: Competition artifacts

    Exit criterion:

    All claims in the submission are supported by archived outputs
    generated by the final code.

    ## Notebook Completion Rule

    A notebook is complete only when:

    1. every cell executes without error;
    2. expected output files exist;
    3. validation assertions pass;
    4. a completion manifest is generated;
    5. the notebook is saved in the repository notebooks directory.
    """
).strip() + "\n"

BUILD_PLAN_PATH.write_text(
    build_plan_content,
    encoding="utf-8",
)


# ------------------------------------------------------------
# 10. Verify every generated artifact
# ------------------------------------------------------------

created_files = [
    ENV_EXAMPLE_PATH,
    LICENSE_PATH,
    RUNTIME_MANIFEST_PATH,
    NOTEBOOK_MANIFEST_PATH,
    REQUIREMENTS_README_PATH,
    BUILD_PLAN_PATH,
]

for created_file in created_files:
    assert created_file.exists(), (
        f"Expected file was not created: {created_file}"
    )

    assert created_file.is_file(), (
        f"Expected path is not a file: {created_file}"
    )

    assert created_file.stat().st_size > 0, (
        f"Created file is empty: {created_file}"
    )


# ------------------------------------------------------------
# 11. Validate runtime and notebook manifest contents
# ------------------------------------------------------------

with RUNTIME_MANIFEST_PATH.open(
    "r",
    encoding="utf-8",
) as file:
    saved_runtime_manifest = json.load(file)

with NOTEBOOK_MANIFEST_PATH.open(
    "r",
    encoding="utf-8",
) as file:
    saved_notebook_manifest = json.load(file)

assert saved_runtime_manifest["project_name"] == "NeuroFHIR-QC"
assert saved_runtime_manifest["data_policy"]["synthetic_fhir_only"] is True
assert saved_runtime_manifest["data_policy"]["real_patient_data_allowed"] is False

assert len(saved_notebook_manifest["notebooks"]) == 11
assert saved_notebook_manifest["notebooks"][0]["number"] == "00"
assert saved_notebook_manifest["notebooks"][-1]["number"] == "10"

notebook_numbers = [
    entry["number"]
    for entry in saved_notebook_manifest["notebooks"]
]

assert len(notebook_numbers) == len(set(notebook_numbers)), (
    "Duplicate notebook numbers were found."
)


# ------------------------------------------------------------
# 12. Display completion summary
# ------------------------------------------------------------

print("=" * 76)
print("✅ Reproducibility metadata created successfully")
print(f"✅ Environment template: {ENV_EXAMPLE_PATH}")
print(f"✅ Open-source license: {LICENSE_PATH}")
print(f"✅ Runtime manifest: {RUNTIME_MANIFEST_PATH}")
print(f"✅ Notebook manifest: {NOTEBOOK_MANIFEST_PATH}")
print(f"✅ Dependency strategy: {REQUIREMENTS_README_PATH}")
print(f"✅ Build plan: {BUILD_PLAN_PATH}")
print(f"🐍 Current Colab Python: {platform.python_version()}")
print("🐳 Production backend target: Python 3.11")
print(f"📓 Planned notebooks registered: {len(notebook_numbers)}")
print("✅ Runtime and data-safety policies validated")
print("➡️ One final foundation audit remains: Notebook 00, Cell 4")
print("=" * 76)

✅ Reproducibility metadata created successfully
✅ Environment template: /content/drive/MyDrive/neurofhir-qc/.env.example
✅ Open-source license: /content/drive/MyDrive/neurofhir-qc/LICENSE
✅ Runtime manifest: /content/drive/MyDrive/neurofhir-qc/runtime_manifest.json
✅ Notebook manifest: /content/drive/MyDrive/neurofhir-qc/notebook_manifest.json
✅ Dependency strategy: /content/drive/MyDrive/neurofhir-qc/requirements/README.md
✅ Build plan: /content/drive/MyDrive/neurofhir-qc/docs/BUILD_PLAN.md
🐍 Current Colab Python: 3.12.13
🐳 Production backend target: Python 3.11
📓 Planned notebooks registered: 11
✅ Runtime and data-safety policies validated
➡️ One final foundation audit remains: Notebook 00, Cell 4


In [5]:
# ============================================================
# NeuroFHIR-QC
# Notebook 00: Project Foundation
# Cell 4: Final foundation audit and completion manifest
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import textwrap


# ------------------------------------------------------------
# 1. Define project paths
# ------------------------------------------------------------

PROJECT_ROOT = Path("/content/drive/MyDrive/neurofhir-qc")

CONFIG_PATH = PROJECT_ROOT / "project_config.json"
NOTEBOOK_MANIFEST_PATH = PROJECT_ROOT / "notebook_manifest.json"

CITATION_PATH = PROJECT_ROOT / "CITATION.cff"
SECURITY_PATH = PROJECT_ROOT / "SECURITY.md"

AUDIT_JSON_PATH = (
    PROJECT_ROOT
    / "evaluation"
    / "results"
    / "notebook_00_foundation_audit.json"
)

AUDIT_MARKDOWN_PATH = (
    PROJECT_ROOT
    / "docs"
    / "FOUNDATION_AUDIT.md"
)


# ------------------------------------------------------------
# 2. Confirm required project files exist
# ------------------------------------------------------------

required_files = [
    PROJECT_ROOT / "README.md",
    PROJECT_ROOT / "LICENSE",
    PROJECT_ROOT / ".gitignore",
    PROJECT_ROOT / ".env.example",
    PROJECT_ROOT / "project_config.json",
    PROJECT_ROOT / "runtime_manifest.json",
    PROJECT_ROOT / "notebook_manifest.json",
    PROJECT_ROOT / "docs" / "SCOPE_LOCK.md",
    PROJECT_ROOT / "docs" / "BUILD_PLAN.md",
    PROJECT_ROOT / "requirements" / "README.md",
]

missing_files = [
    str(path)
    for path in required_files
    if not path.exists()
]

if missing_files:
    raise FileNotFoundError(
        "Notebook 00 foundation is incomplete.\n"
        "The following required files are missing:\n\n"
        + "\n".join(missing_files)
    )


# ------------------------------------------------------------
# 3. Confirm required repository directories exist
# ------------------------------------------------------------

required_directories = [
    "notebooks",
    "docs",
    "requirements",
    "data/synthetic_fhir",
    "data/sample_biomarkers",
    "data/sample_images",
    "data/sample_masks",
    "backend/app/api",
    "backend/app/services",
    "backend/app/fhir_builders",
    "evaluation/results",
    "scripts",
    "tests/unit",
    "tests/integration",
    "submission/fhir_resources",
    "submission/screenshots",
]

missing_directories = [
    directory
    for directory in required_directories
    if not (PROJECT_ROOT / directory).exists()
]

if missing_directories:
    raise FileNotFoundError(
        "Required repository directories are missing:\n\n"
        + "\n".join(missing_directories)
    )


# ------------------------------------------------------------
# 4. Create CITATION.cff
# ------------------------------------------------------------

citation_content = textwrap.dedent(
    """
    cff-version: 1.2.0
    message: "Please cite this software using the metadata below."
    title: "NeuroFHIR-QC"
    type: software
    version: 0.1.0

    authors:
      - family-names: "Basu"
        given-names: "Sanghati"

    abstract: >
      NeuroFHIR-QC is a human-in-the-loop FHIR R4 research application
      for transforming longitudinal brain-tumor or lesion measurements
      into quality-scored, provenance-aware and reviewable FHIR evidence.

    keywords:
      - FHIR
      - neuroinformatics
      - medical imaging
      - artificial intelligence
      - provenance
      - human-in-the-loop
      - quality control
      - longitudinal biomarkers

    license: MIT

    repository-code: >
      https://github.com/REPLACE_WITH_GITHUB_USERNAME/neurofhir-qc
    """
).strip() + "\n"

CITATION_PATH.write_text(
    citation_content,
    encoding="utf-8",
)


# ------------------------------------------------------------
# 5. Create SECURITY.md
# ------------------------------------------------------------

security_content = textwrap.dedent(
    """
    # Security Policy

    ## Supported Status

    NeuroFHIR-QC is an academic research implementation under active
    development.

    It is not a clinically deployed system, diagnostic medical device
    or treatment-recommendation application.

    ## Data Restrictions

    The repository must contain only:

    - synthetic FHIR records;
    - public de-identified neuroimaging data;
    - generated demonstration artifacts;
    - non-sensitive evaluation results.

    The repository must never contain:

    - protected health information;
    - real patient identifiers;
    - access tokens;
    - OAuth secrets;
    - passwords;
    - private encryption keys;
    - institutional credentials;
    - restricted clinical imaging data.

    ## Environment Variables

    Secrets must be stored in a local `.env` file or an approved secret
    manager.

    The `.env` file must never be committed to GitHub.

    Only `.env.example`, containing placeholder values, may be included
    in the public repository.

    ## FHIR Write-Back Safety

    FHIR write-back must remain disabled by default.

    The application must require an explicit configuration change before
    transaction Bundles can be submitted to a FHIR server.

    Development and competition testing must use a designated sandbox or
    local HAPI FHIR server.

    ## AI Result Safety

    Every AI-generated clinical-result Observation begins with:

    `Observation.status = preliminary`

    It may become final only after an explicit human-review action.

    Low-confidence or unstable results must remain preliminary and must
    be routed for rejection or correction.

    ## Reporting a Security Concern

    Do not open a public GitHub issue containing sensitive information.

    Contact the repository owner privately and provide:

    - a description of the concern;
    - reproduction steps;
    - affected files or components;
    - potential impact;
    - suggested mitigation, when available.
    """
).strip() + "\n"

SECURITY_PATH.write_text(
    security_content,
    encoding="utf-8",
)


# ------------------------------------------------------------
# 6. Add .gitkeep files to empty directories
# ------------------------------------------------------------

gitkeep_files_created = []

for directory_name in required_directories:
    directory_path = PROJECT_ROOT / directory_name

    visible_contents = [
        item
        for item in directory_path.iterdir()
        if item.name != ".gitkeep"
    ]

    if not visible_contents:
        gitkeep_path = directory_path / ".gitkeep"
        gitkeep_path.touch(exist_ok=True)
        gitkeep_files_created.append(
            gitkeep_path.relative_to(PROJECT_ROOT).as_posix()
        )


# ------------------------------------------------------------
# 7. Load and validate the project configuration
# ------------------------------------------------------------

with CONFIG_PATH.open("r", encoding="utf-8") as file:
    project_config = json.load(file)

assert project_config["project_name"] == "NeuroFHIR-QC"

assert set(project_config["required_demo_cases"]) == {
    "stable",
    "progression",
    "low-confidence",
}

assert set(project_config["core_fhir_resources"]) == {
    "Patient",
    "Condition",
    "ImagingStudy",
    "Observation",
    "DiagnosticReport",
    "Device",
    "Provenance",
    "Task",
    "Bundle",
}

assert "preliminary" in project_config["safety_rule"].lower()
assert "human" in project_config["safety_rule"].lower()
assert "review" in project_config["safety_rule"].lower()


# ------------------------------------------------------------
# 8. Validate environment and secret protections
# ------------------------------------------------------------

env_example_text = (
    PROJECT_ROOT / ".env.example"
).read_text(encoding="utf-8")

gitignore_text = (
    PROJECT_ROOT / ".gitignore"
).read_text(encoding="utf-8")

assert "SYNTHETIC_DATA_ONLY=true" in env_example_text
assert "ALLOW_FHIR_WRITEBACK=false" in env_example_text
assert "SMART_CLIENT_ID=" in env_example_text

assert ".env" in gitignore_text
assert "*.key" in gitignore_text
assert "*.pem" in gitignore_text
assert "data/private/" in gitignore_text


# ------------------------------------------------------------
# 9. Mark Notebook 00 as completed
# ------------------------------------------------------------

with NOTEBOOK_MANIFEST_PATH.open(
    "r",
    encoding="utf-8",
) as file:
    notebook_manifest = json.load(file)

notebook_00_found = False
completion_timestamp = datetime.now(timezone.utc).isoformat()

for notebook_entry in notebook_manifest["notebooks"]:
    if notebook_entry["number"] == "00":
        notebook_entry["status"] = "completed"
        notebook_entry["completed_utc"] = completion_timestamp
        notebook_entry["validation_status"] = "passed"
        notebook_00_found = True
        break

assert notebook_00_found, (
    "Notebook 00 was not found in notebook_manifest.json."
)

with NOTEBOOK_MANIFEST_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        notebook_manifest,
        file,
        indent=2,
    )


# ------------------------------------------------------------
# 10. Define file-hashing function
# ------------------------------------------------------------

def calculate_sha256(file_path: Path) -> str:
    """
    Calculate a SHA-256 checksum for reproducibility auditing.
    """
    sha256 = hashlib.sha256()

    with file_path.open("rb") as file:
        while True:
            block = file.read(1024 * 1024)

            if not block:
                break

            sha256.update(block)

    return sha256.hexdigest()


# ------------------------------------------------------------
# 11. Build repository file inventory
# ------------------------------------------------------------

excluded_audit_files = {
    AUDIT_JSON_PATH.resolve(),
    AUDIT_MARKDOWN_PATH.resolve(),
}

repository_files = []

for file_path in sorted(PROJECT_ROOT.rglob("*")):
    if not file_path.is_file():
        continue

    if file_path.resolve() in excluded_audit_files:
        continue

    relative_path = file_path.relative_to(PROJECT_ROOT).as_posix()

    repository_files.append(
        {
            "path": relative_path,
            "size_bytes": file_path.stat().st_size,
            "sha256": calculate_sha256(file_path),
        }
    )


# ------------------------------------------------------------
# 12. Create formal audit checks
# ------------------------------------------------------------

audit_checks = {
    "project_root_exists": PROJECT_ROOT.exists(),
    "required_files_present": len(missing_files) == 0,
    "required_directories_present": len(missing_directories) == 0,
    "project_name_valid": (
        project_config["project_name"] == "NeuroFHIR-QC"
    ),
    "three_demo_cases_registered": (
        len(project_config["required_demo_cases"]) == 3
    ),
    "nine_core_fhir_resources_registered": (
        len(project_config["core_fhir_resources"]) == 9
    ),
    "human_review_required": (
        "human" in project_config["safety_rule"].lower()
        and "review" in project_config["safety_rule"].lower()
    ),
    "preliminary_status_required": (
        "preliminary" in project_config["safety_rule"].lower()
    ),
    "synthetic_data_mode_enabled": (
        "SYNTHETIC_DATA_ONLY=true" in env_example_text
    ),
    "fhir_writeback_disabled_by_default": (
        "ALLOW_FHIR_WRITEBACK=false" in env_example_text
    ),
    "secret_files_gitignored": (
        ".env" in gitignore_text
        and "*.key" in gitignore_text
        and "*.pem" in gitignore_text
    ),
    "citation_metadata_present": CITATION_PATH.exists(),
    "security_policy_present": SECURITY_PATH.exists(),
    "notebook_00_marked_completed": any(
        entry["number"] == "00"
        and entry["status"] == "completed"
        and entry["validation_status"] == "passed"
        for entry in notebook_manifest["notebooks"]
    ),
}

failed_checks = [
    check_name
    for check_name, passed
    in audit_checks.items()
    if not passed
]

assert not failed_checks, (
    "Foundation audit failed for the following checks: "
    f"{failed_checks}"
)


# ------------------------------------------------------------
# 13. Create machine-readable completion manifest
# ------------------------------------------------------------

foundation_audit = {
    "project_name": "NeuroFHIR-QC",
    "notebook": (
        "00_NeuroFHIR_QC_Project_Foundation.ipynb"
    ),
    "notebook_status": "completed",
    "validation_status": "passed",
    "completed_utc": completion_timestamp,
    "project_root": str(PROJECT_ROOT),
    "required_directory_count": len(required_directories),
    "repository_file_count": len(repository_files),
    "gitkeep_files_created": gitkeep_files_created,
    "audit_checks": audit_checks,
    "repository_inventory": repository_files,
    "next_notebook": (
        "01_NeuroFHIR_QC_Synthetic_FHIR_Foundation.ipynb"
    ),
}

AUDIT_JSON_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)

with AUDIT_JSON_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        foundation_audit,
        file,
        indent=2,
    )


# ------------------------------------------------------------
# 14. Create human-readable audit report
# ------------------------------------------------------------

audit_rows = "\n".join(
    f"| {check_name.replace('_', ' ').title()} | "
    f"{'PASS' if passed else 'FAIL'} |"
    for check_name, passed in audit_checks.items()
)

audit_markdown_content = textwrap.dedent(
    f"""
    # NeuroFHIR-QC Foundation Audit

    **Notebook:** `00_NeuroFHIR_QC_Project_Foundation.ipynb`

    **Status:** Completed

    **Validation:** Passed

    **Completed UTC:** {completion_timestamp}

    ## Audit Results

    | Check | Result |
    |---|---|
    {audit_rows}

    ## Repository Summary

    - Required directories: {len(required_directories)}
    - Registered files: {len(repository_files)}
    - Core FHIR resources: 9
    - Required demo cases: 3
    - Synthetic FHIR data only: enabled
    - FHIR write-back default: disabled
    - Human review before finalization: required

    ## Locked Demonstration Cases

    1. Stable
    2. Progression
    3. Low confidence

    ## Next Notebook

    `01_NeuroFHIR_QC_Synthetic_FHIR_Foundation.ipynb`
    """
).strip() + "\n"

AUDIT_MARKDOWN_PATH.write_text(
    audit_markdown_content,
    encoding="utf-8",
)


# ------------------------------------------------------------
# 15. Final output verification
# ------------------------------------------------------------

final_required_files = [
    CITATION_PATH,
    SECURITY_PATH,
    AUDIT_JSON_PATH,
    AUDIT_MARKDOWN_PATH,
]

for file_path in final_required_files:
    assert file_path.exists(), (
        f"Final artifact was not created: {file_path}"
    )

    assert file_path.stat().st_size > 0, (
        f"Final artifact is empty: {file_path}"
    )


with AUDIT_JSON_PATH.open(
    "r",
    encoding="utf-8",
) as file:
    saved_audit = json.load(file)

assert saved_audit["notebook_status"] == "completed"
assert saved_audit["validation_status"] == "passed"
assert all(saved_audit["audit_checks"].values())


# ------------------------------------------------------------
# 16. Display audit summary
# ------------------------------------------------------------

print("=" * 76)
print("✅ NeuroFHIR-QC foundation audit passed")
print(f"✅ Audit checks passed: {len(audit_checks)}")
print(f"✅ Repository files inventoried: {len(repository_files)}")
print(f"✅ Empty directories preserved for GitHub: {len(gitkeep_files_created)}")
print(f"✅ Citation metadata: {CITATION_PATH}")
print(f"✅ Security policy: {SECURITY_PATH}")
print(f"✅ Machine-readable audit: {AUDIT_JSON_PATH}")
print(f"✅ Human-readable audit: {AUDIT_MARKDOWN_PATH}")
print("✅ Notebook 00 completed")
print("=" * 76)

✅ NeuroFHIR-QC foundation audit passed
✅ Audit checks passed: 14
✅ Repository files inventoried: 26
✅ Empty directories preserved for GitHub: 14
✅ Citation metadata: /content/drive/MyDrive/neurofhir-qc/CITATION.cff
✅ Security policy: /content/drive/MyDrive/neurofhir-qc/SECURITY.md
✅ Machine-readable audit: /content/drive/MyDrive/neurofhir-qc/evaluation/results/notebook_00_foundation_audit.json
✅ Human-readable audit: /content/drive/MyDrive/neurofhir-qc/docs/FOUNDATION_AUDIT.md
✅ Notebook 00 completed
